# Fase 09 — Evaluación crítica de la calidad de la evidencia

Esta fase evalúa la calidad **a nivel de fuente** mediante criterios diferenciados por tipo documental. Los 97 hallazgos validados de la fase 08 heredan posteriormente la evaluación de su fuente.

Principios:

- calidad metodológica, autoridad jurídica, relevancia y transferibilidad son dimensiones distintas;
- prestigio de revista, citaciones o reputación institucional no sustituyen la evaluación;
- una actividad o producto de proyecto no es un resultado o impacto;
- recomendaciones, escenarios y proyecciones no son evidencia observada de efectividad;
- todo puntaje debe conservar justificación y localizador de página;
- las evaluaciones asistidas permanecen `not_reviewed` hasta validación humana.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from evidence_review.quality_appraisal import (
    appraisal_summary,
    build_finding_quality_link,
    build_source_quality_summary,
    export_quality_appraisal_prompts_jsonl,
    initialise_quality_appraisal_sheet,
    load_quality_appraisal_config,
    merge_quality_appraisals,
    read_csv_robust,
    read_quality_appraisal_responses_jsonl,
    select_quality_appraisal_packets,
    split_quality_appraisal_outputs,
    validate_quality_appraisal_sheet,
)

CONFIG_PATH = ROOT / "config" / "quality_appraisal.yml"
config = load_quality_appraisal_config(
    CONFIG_PATH,
    project_root=ROOT,
)
paths = config["paths"]

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Configuration: {CONFIG_PATH.relative_to(ROOT)}")

## 1. Cargar fuentes, hallazgos validados y corpus

La unidad de evaluación es la fuente incluida en la fase 07. Los hallazgos validados se utilizan para contabilizar la evidencia aportada por cada fuente, mientras que el corpus aporta texto trazable para construir los paquetes de evaluación.

In [ ]:
sources_path = ROOT / paths["sources_csv"]
findings_path = ROOT / paths["findings_csv"]
corpus_path = ROOT / paths["corpus_csv"]
working_path = ROOT / paths["appraisal_working_csv"]

sources, sources_encoding = read_csv_robust(sources_path)
findings, findings_encoding = read_csv_robust(findings_path)
corpus, corpus_encoding = read_csv_robust(corpus_path)

existing = pd.DataFrame()
if working_path.exists():
    existing, working_encoding = read_csv_robust(working_path)
else:
    working_encoding = "not_loaded"

print(f"Included sources: {len(sources)} [{sources_encoding}]")
print(f"Validated findings: {len(findings)} [{findings_encoding}]")
print(f"Corpus chunks: {len(corpus)} [{corpus_encoding}]")
print(f"Existing appraisal rows: {len(existing)} [{working_encoding}]")

## 2. Inicializar la hoja de evaluación

Se crea una fila por fuente. Las evaluaciones humanas existentes se preservan por `source_id`. El notebook no sobrescribe una hoja ya existente durante la inicialización.

In [ ]:
sheet = initialise_quality_appraisal_sheet(
    sources,
    findings,
    config,
    existing,
)

if not working_path.exists():
    sheet.to_csv(
        working_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Created: {working_path.relative_to(ROOT)}")
else:
    print(
        "Existing quality_appraisal_working.csv preserved; "
        "initialization did not overwrite it."
    )

display(appraisal_summary(sheet))
display(
    sheet[
        [
            "source_id",
            "title",
            "source_type",
            "appraisal_domain",
            "validated_findings_count",
            "appraisal_status",
            "human_validation_status",
        ]
    ].head(20)
)

## 3. Construir paquetes y prompts auditables

Cada paquete prioriza métodos, datos, resultados, incertidumbre, sensibilidad, limitaciones, monitoreo, evaluación e implementación. Esta sección no llama a ningún modelo externo.

In [ ]:
packets = select_quality_appraisal_packets(
    corpus,
    sources,
    config,
)

packets_path = ROOT / paths["packets_csv"]
packets.to_csv(
    packets_path,
    index=False,
    encoding="utf-8-sig",
)

prompts_path = ROOT / paths["prompts_jsonl"]
export_quality_appraisal_prompts_jsonl(
    sheet,
    packets,
    config,
    prompts_path,
)

print(f"Sources represented in packets: {packets['source_id'].nunique()}")
print(f"Packet rows: {len(packets)}")
print(f"Prompts generated: {len(sheet)}")
print(f"Saved: {prompts_path.relative_to(ROOT)}")
display(
    packets.groupby("source_id")
    .agg(packet_chunks=("chunk_id", "size"),
         packet_characters=("character_count", "sum"))
    .reset_index()
)

## 4. Importar evaluaciones asistidas opcionales

El archivo esperado es `quality_appraisal_model_responses.jsonl`, con una evaluación por fuente. La importación está desactivada por defecto. Las filas humanas `accepted`, `corrected` o `rejected` están protegidas.

In [ ]:
IMPORT_MODEL_RESPONSES = False
OVERWRITE_HUMAN_VALIDATED = False

responses_path = ROOT / paths["model_responses_jsonl"]

if IMPORT_MODEL_RESPONSES:
    if not responses_path.exists():
        raise FileNotFoundError(responses_path)

    incoming = read_quality_appraisal_responses_jsonl(
        responses_path,
        sheet,
        config,
    )
    sheet = merge_quality_appraisals(
        sheet,
        incoming,
        config,
        overwrite_human_validated=OVERWRITE_HUMAN_VALIDATED,
    )
    sheet.to_csv(
        working_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Imported source appraisals: {len(incoming)}")
    print(f"Updated: {working_path.relative_to(ROOT)}")
else:
    print(
        "Model-response import disabled. Set "
        "IMPORT_MODEL_RESPONSES = True only after reviewing the JSONL."
    )

## 5. Recargar y validar la hoja revisada

La hoja se recarga desde disco inmediatamente antes de validar. Esto evita exportar una versión antigua mantenida en memoria.

In [ ]:
sheet, reloaded_encoding = read_csv_robust(working_path)

print(
    f"Reloaded appraisals from: "
    f"{working_path.relative_to(ROOT)} [{reloaded_encoding}]"
)
print(
    sheet["human_validation_status"]
    .value_counts(dropna=False)
)

issues = validate_quality_appraisal_sheet(
    sheet,
    config,
)

issues_path = ROOT / paths["issues_csv"]
issues.to_csv(
    issues_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Validation issues: {len(issues)}")
display(issues.head(100))
display(appraisal_summary(sheet))

## 6. Exportar evaluaciones y vincularlas con los hallazgos

`quality_appraisal_validated.csv` contiene únicamente evaluaciones `completed` con validación humana `accepted` o `corrected`. El vínculo con los hallazgos utiliza solo esas evaluaciones validadas.

In [ ]:
# Recarga defensiva justo antes de exportar.
sheet, _ = read_csv_robust(working_path)

groups = split_quality_appraisal_outputs(sheet)
finding_quality = build_finding_quality_link(
    findings,
    groups["validated"],
)
source_summary = build_source_quality_summary(sheet)
flow = appraisal_summary(sheet)

output_frames = {
    paths["appraisal_csv"]: groups["appraisal"],
    paths["validated_appraisal_csv"]: groups["validated"],
    paths["pending_validation_csv"]: groups["pending_validation"],
    paths["rejected_appraisal_csv"]: groups["rejected"],
    paths["finding_quality_link_csv"]: finding_quality,
    paths["source_summary_csv"]: source_summary,
    paths["flow_summary_csv"]: flow,
}

for relative_path, frame in output_frames.items():
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"{path.relative_to(ROOT)}: {len(frame)}")

display(flow)
display(source_summary)

## Criterio para avanzar a la fase 10

La síntesis y evaluación de transferibilidad puede comenzar cuando:

1. `Validation issues = 0`;
2. las 16 fuentes tienen evaluación `completed` o una justificación documentada de texto insuficiente;
3. las evaluaciones utilizadas están `accepted` o `corrected`;
4. cada criterio puntuado conserva nota y localizador;
5. las puntuaciones totales, normalizadas y categorías son consistentes;
6. los 97 hallazgos están vinculados con evaluaciones de calidad validadas;
7. calidad metodológica, autoridad, relevancia y transferibilidad permanecen separadas.

La siguiente fase será:

```text
notebooks/10_synthesise_transferability.ipynb
```